In [1]:
import os
import pandas as pd
import numpy as np
import random
import torch
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification


# Warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Text processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [2]:
# Initialize constant variables.
INPUT_FOLDER = 'data'
OUTPUT_FOLDER = 'dataProcessed'

In [3]:
# Random States
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

## Text Processing

In [4]:
stop_words = set(stopwords.words("english"))

In [5]:
# Make function to remove punctuation, make lowercase, remove stopwords, punctuation, remove documents with less than or equal to 1 token.
def preprocessing(notes, min_words=1):
    cleaned_notes = []

    for note in notes:
        tokens = word_tokenize(note, language='english')
        tokens = [token.lower() for token in tokens]
        tokens = [token for token in tokens if token.isalpha() and token not in stop_words]

        if len(tokens) >= min_words:
            cleaned_notes.append(" ".join(tokens))

    return cleaned_notes

In [6]:
# Process all T1 files (this can be adjusted later for T2 also). 
# Minor processing of data points to make separate date and time columns where applicable and removing NaN values.
all_data = {}
for folder in os.listdir(f'./{INPUT_FOLDER}/'):
    if '.' not in folder:
        for sub_folder in os.listdir(f'./{INPUT_FOLDER}/{folder}'):
            if '.' not in sub_folder and 'T1' in sub_folder:
                for sub_sub_folder in os.listdir(f'./{INPUT_FOLDER}/{folder}/{sub_folder}'):
                    if '.' not in sub_sub_folder: 
                        if 'P15' not in sub_sub_folder:
                            # -------- Carer Notes --------
                            carer_notes_1 = pd.read_excel(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/carerNotes1_{sub_sub_folder.split(' ')[0]}.xlsx', skiprows= 3, header=[0, 1])
                            carer_notes_2 = pd.read_excel(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/carerNotes2_{sub_sub_folder.split(' ')[0]}.xlsx', skiprows= 3, header=[0, 1])
                            carer_notes_1.columns = [
                            "_".join([str(x) for x in col if "Unnamed" not in str(x)]).strip()
                            for col in carer_notes_1.columns
                            ]
                            carer_notes_2.columns = [
                            "_".join([str(x) for x in col if "Unnamed" not in str(x)]).strip()
                            for col in carer_notes_2.columns
                            ]
                            carer_notes_1 = carer_notes_1[['Date', 'Activity']]
                            carer_notes_2 = carer_notes_2[['Date', 'Activity']]
                            carer_notes = pd.concat([carer_notes_1, carer_notes_2])
                            carer_notes['datetime'] = pd.to_datetime(carer_notes['Date'], format='%d/%m/%y %H:%M')
                            carer_notes['Date'] = carer_notes['datetime'].dt.date
                            carer_notes['Time'] = carer_notes['datetime'].dt.time
                            carer_notes = carer_notes[['Date', 'Time', 'Activity']]
                            carer_notes = carer_notes.dropna(subset=['Date', 'Time', 'Activity'])
                            carer_notes['Carer Note'] = carer_notes['Activity'].values.tolist()
                            carer_notes = carer_notes.drop(columns=['Activity'])
                        else:
                            # -------- Carer Notes --------
                            carer_notes = pd.read_excel(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/carerNotes_{sub_sub_folder.split(' ')[0]}.xlsx', skiprows= 3, header=[0, 1])
                            carer_notes.columns = [
                            "_".join([str(x) for x in col if "Unnamed" not in str(x)]).strip()
                            for col in carer_notes.columns
                            ]
                            carer_notes = carer_notes[['Date', 'Activity']]
                            carer_notes['datetime'] = pd.to_datetime(carer_notes['Date'], format='%d/%m/%y %H:%M')
                            carer_notes['Date'] = carer_notes['datetime'].dt.date
                            carer_notes['Time'] = carer_notes['datetime'].dt.time
                            carer_notes = carer_notes[['Date', 'Time', 'Activity']]
                            carer_notes = carer_notes.dropna(subset=['Date', 'Time', 'Activity'])
                            carer_notes['Carer Note'] = carer_notes['Activity'].values.tolist()
                            carer_notes = carer_notes.drop(columns=['Activity'])
                        
                        # -------- Nurse Notes --------
                        daily_nurse = pd.read_excel(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/dailyNurseNotes_{sub_sub_folder.split(' ')[0]}.xlsx')
                        daily_nurse = daily_nurse[['Date', 'Time', 'Note']]
                        daily_nurse['Nurse Note'] = daily_nurse['Note'].values.tolist()
                        daily_nurse = daily_nurse.dropna(subset=['Note', 'Date', 'Time'])
                        daily_nurse = daily_nurse.drop(columns=['Note'])
                        # -------- Monthly --------
                        monthly = pd.read_excel(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/monthly_{sub_sub_folder.split(' ')[0]}.xlsx')
                        monthly = monthly.drop(columns=['Resident Study Number'])
                        # -------- Multi-Disciplinary Notes --------
                        multi = pd.read_excel(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/multiDisciplinaryNotes_{sub_sub_folder.split(' ')[0]}.xlsx')
                        multi['Multi-Disciplinary Note'] = multi['Note'].values.tolist()
                        multi = multi.drop(columns=['Resident Study Number', 'Delirium Indicated', 'Note', 'Note Type'])
                        multi = multi.dropna(subset=['Multi-Disciplinary Note'])
                        # -------- Quarterly --------
                        quarterly = pd.read_excel(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/quarterly_{sub_sub_folder.split(' ')[0]}.xlsx')
                        quarterly = quarterly.drop(columns=['Resident Study Number'])
                        # Big Concatenation 
                        big_df = pd.concat([carer_notes, daily_nurse, multi, monthly, quarterly])
                        cleaned_dates = []
                        for date in big_df['Date']:
                            date = (str(date)).strip()
                            cleaned_dates.append(date.split(' ')[0])
                        big_df['Date'] = cleaned_dates
                        big_df['DateTime'] = pd.to_datetime(big_df['Date'].astype(str) + ' ' + big_df['Time'].astype(str), format='mixed')
                        big_columns = [(column.split('\n')[0]).replace("'", '') for column in big_df.columns]
                        big_df.columns = big_columns
                        big_df = big_df.sort_values('DateTime')
                        # -------- Meds --------
                        meds = pd.read_excel(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/meds_{sub_sub_folder.split(' ')[0]}.xlsx')
                        meds = meds.dropna(subset=['Medications in Use in previous 6 to 9 months'])
                        meds = meds[['Medications in Use in previous 6 to 9 months', 'Regular, PRN, or short course', 'Date started (all meds)', 'Date discontinued']]
                        meds_columns = [(column.split('\n')[0]).replace("'", '') for column in meds.columns]
                        meds.columns = meds_columns
                        meds.columns = [
                            "".join([str(x) for x in col if "Unnamed" not in str(x)]).strip()
                            for col in meds.columns
                            ]
                        meds = meds.drop(columns=[column for column in meds.columns if 'Unnamed' in column])
                        # -------- Demographics --------
                        t1 = pd.read_excel(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/t1_{sub_sub_folder.split(' ')[0]}.xlsx')   
                        t1_columns = [(column.split('\n')[0]).replace("'", '') for column in t1.columns]
                        t1.columns = t1_columns
                        t1.columns = [
                            "".join([str(x) for x in col if "Unnamed" not in str(x)]).strip()
                            for col in t1.columns
                            ]
                        t1 = t1.drop(columns=[column for column in t1.columns if 'Unnamed' in column])
                        all_data[sub_sub_folder.split(' ')[0]] = {'Temporal Information': big_df, 'Medications': meds, 'Demographics': t1}                      


In [7]:
for patient in all_data:
    temp_notes = preprocessing(all_data[patient]['Temporal Information']['Nurse Note'].dropna().values.tolist())
    break

In [8]:
# https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english

from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)

text = "These are cats."
result = classifier(text)
print(result)  

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.8794072866439819}]


In [9]:
# https://huggingface.co/minuva/MiniLMv2-goemotions-v2

from transformers import pipeline
import torch

# possible positive classes are: admiration, amusement, approval, caring, desire, excitement, gratitude, joy, love, optimism, pride, relief
# possible negative classes are: anger, annoyance, disappointment, disgust, embarrassment, fear, grief, nervousness, remorse, sadness
# possible ambiguous classes are: confusion, curiosity, realization, surprise

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model="minuva/MiniLMv2-goemotions-v2",
    device=device,
)

text = "I realize!"
result = classifier(text)
print(result)  

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[{'label': 'realization', 'score': 0.9614923000335693}]


In [10]:
# https://huggingface.co/agentlans/snowflake-arctic-xs-grammar-classifier

from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model="agentlans/snowflake-arctic-xs-grammar-classifier",
    device=device,
)

text = "I absolutely loved these cats."
result = classifier(text)
print(result)  

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[{'label': 'grammatical', 'score': 0.9142002463340759}]


In [11]:
# https://huggingface.co/nikk118/minilm-finetuned-emotion

from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model="nikk118/minilm-finetuned-emotion",
    device=device,
)

text = "I absolutely loved these cats."
result = classifier(text)
print(result)  

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'love', 'score': 0.5610654354095459}]


In [12]:
# https://huggingface.co/techhy/mindtrack-mental-health-analyzer

from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model="techhy/mindtrack-mental-health-analyzer",
    device=device,
)

text = "Resident wants to be alone."
result = classifier(text)
print(result)  

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'RISK', 'score': 0.9948431253433228}]


In [13]:
# https://huggingface.co/Kebinnuil/suicidal_detection_model

from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model="Kebinnuil/suicidal_detection_model",
    device=device,
)

text = "Resident wants to be alone."
result = classifier(text)
print(result)  

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'Suicidal', 'score': 0.7808234691619873}]


In [14]:
# https://huggingface.co/vhdm/clinicalbert-ms-autoimmune-neuro
# label 1 is detection label 0 is not present (detection of autoimmune neurological disease signals)

from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model="vhdm/clinicalbert-ms-autoimmune-neuro",
    device=device,
)

text = "Patient reports numbness in lower limbs."
result = classifier(text)
print(result)  

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.571586549282074}]


In [15]:
# https://huggingface.co/Clinical-Emotisupport/NLP-Clinical-Emotisupport

from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model="Clinical-Emotisupport/NLP-Clinical-Emotisupport",
    device=device,
)

text = "Patient reports numbness in lower limbs."
result = classifier(text)
print(result)  

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'confusion', 'score': 0.6328957676887512}]
